# 7회차 실습: 열공간·rank·차원정리를 **눈으로**

교안과 같은 행렬 $B=\begin{pmatrix}1&2&0\\2&4&1\\3&6&1\end{pmatrix}$ ($\mathbf{b}_2=2\mathbf{b}_1$, rank 2)로 본다.

1. **col(B)** = $B$가 도달할 수 있는 영역 (rank 2라 평면)
2. **차원정리** rank + nullity = n 을 직교 분해로
3. **AI 발췌**: 유효 rank가 낮다 → 저계수 근사·LoRA


In [ ]:
!pip install -q koreanize-matplotlib
import numpy as np, matplotlib.pyplot as plt, koreanize_matplotlib  # noqa
from mpl_toolkits.mplot3d import Axes3D  # noqa
CR, NV, GD = '#862633', '#22386B', '#9A6B00'
B = np.array([[1., 2, 0], [2, 4, 1], [3, 6, 1]])
rng = np.random.default_rng(0)


## ① col(B): 도달할 수 있는 영역

모든 $B\mathbf{x}$를 찍으면 $\mathbb{R}^3$ 전체가 아니라 **평면**만 채운다. rank가 2이기 때문이다 (세 열 중 $\mathbf{b}_2=2\mathbf{b}_1$라 실제 방향은 2개).


In [ ]:
X = rng.uniform(-2, 2, (3, 800))
Y = B @ X                                   # 모든 도달점 Bx
fig = plt.figure(figsize=(6, 5.2)); ax = fig.add_subplot(111, projection='3d')
ax.scatter(Y[0], Y[1], Y[2], s=4, color=CR, alpha=.2)
ax.set_title(f'col(B): 모든 Bx → 평면 (rank = {np.linalg.matrix_rank(B)})'); ax.view_init(18, -60)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z'); plt.tight_layout(); plt.show()


## ② 차원정리 = 정의역의 직교 분해

정의역 $\mathbb{R}^3$은 **행공간(평면, 2차원) $\perp$ Null space(직선, 1차원)**으로 갈라진다. $B$는 Null 직선을 원점으로 보내고 행공간 평면만 col(B)로 살려 보낸다: $\underbrace{2}_{rank}+\underbrace{1}_{nullity}=3$.


In [ ]:
n = np.array([-2., 1, 0])                     # N(B) 방향
r1, r2 = B[0], B[1] - 2*B[0]                  # 행공간 두 방향(독립)
g = np.linspace(-1, 1, 2); S, T = np.meshgrid(g, g)
plane = r1[:, None, None]*S + r2[:, None, None]*T
fig = plt.figure(figsize=(6, 5.4)); ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(plane[0], plane[1], plane[2], alpha=.2, color=NV)
ln = n[:, None]*np.linspace(-1.5, 1.5, 2)[None, :]
ax.plot(ln[0], ln[1], ln[2], color=CR, lw=3, label='Null space (1차원)')
ax.text(*r1*0.8, ' 행공간 평면(2차원)', color=NV)
ax.set_title('정의역 R³ = 행공간 + Null space 직교 분해 (2+1=3)'); ax.legend(); ax.view_init(20, -55)
plt.tight_layout(); plt.show()


## ③ 직교를 숫자로 확인

행공간 $\perp$ Null space: $B$의 **모든 행**이 Null 방향 $(-2,1,0)$과 수직이다.


In [ ]:
print('B @ n =', np.round(B @ n, 6), ' (Null space 정의: 0)')
for i, row in enumerate(B):
    print(f'행{i+1} {row} · n = {row @ n:.0f}')   # 모두 0 → 행공간이 Null과 수직


## ④ AI 응용: 유효 **rank**가 낮다 → 저계수·LoRA

실제 가중치·데이터 행렬은 겉보기 크기(예: 100×50)보다 **유효 rank가 훨씬 낮다** — 독립인 방향이 몇 개 안 된다. 이 사실을 이용해 큰 행렬을 **작은 두 행렬의 곱**으로 근사하는 것이 **LoRA·저계수 근사**다 (정식 분해 SVD는 Part 2).


In [ ]:
U = rng.normal(0, 1, (100, 2)); V = rng.normal(0, 1, (2, 50))
W = U @ V + rng.normal(0, 0.01, (100, 50))    # 사실상 방향 2개 + 미세 잡음
print('W shape =', W.shape, ' (겉보기 rank 최대 50)')
print('유효 rank =', np.linalg.matrix_rank(W, tol=1.0), ' → 독립 방향은 사실상 2개뿐')


## 정리

- **col(B)** = 도달 영역, 차원 = **rank**. rank 2면 $\mathbb{R}^3$의 평면.
- **차원정리**: 정의역 = 행공간 $\perp$ Null space, $\dim = rank + nullity = n$.
- 실데이터는 **유효 rank가 낮다** → 저계수 근사·LoRA·압축의 근거 (Part 2 SVD).
